# 第二章：保证金和爆仓 / Chapter 2: Margin and Liquidation

## 你将学到 / What You Will Learn
- 什么是保证金（你的"押金"）/ What margin is (your "deposit")
- 怎么计算保证金 / How to compute margin
- 什么是爆仓（你最不想遇到的事）/ What liquidation is (the thing you never want to meet)
- 怎么算出"价格跌/涨到多少你会爆仓" / How to work out the exact price at which you get liquidated

---

## 2.1 保证金 = 你的押金 / 2.1 Margin = Your Security Deposit

> 你想交易价值 10万美元 的黄金，但你账户只有 1万美元。
> 平台说："没关系，你押 1万 就行，剩下的我借你。"
> 这个"押金"就是**保证金**。
>
> You want to trade USD 100 000 of gold, but your account only has USD 10 000.
> The broker says: "No problem, put up 10 000 as deposit — I will lend you the rest."
> That deposit is the **margin**.

### 保证金公式 / The Margin Formula

$$\text{保证金 / Margin} = \frac{\text{手数 / Lots} \times \text{合约量 / ContractSize} \times \text{价格 / Price}}{\text{杠杆 / Leverage}}$$

拆解一下：

Breaking it down:

- **手数 / Lots**：你要交易多少"手"（1手是标准交易单位）/ How many "lots" you trade (1 lot is the standard unit)
- **合约量 / Contract size**：1手代表多少东西（黄金1手=100盎司，外汇1手=10万单位）/ How much 1 lot actually is (gold: 1 lot = 100 oz; FX: 1 lot = 100 000 units)
- **价格 / Price**：当前市价 / The current market price
- **杠杆 / Leverage**：平台借你多少倍（杠杆100 = 你只出1%）/ How much the broker lends you (100× leverage → you only post 1 %)

## 2.2 各产品的合约量速查表 / 2.2 Contract-Size Cheat Sheet

| 产品 / Product | 1手 = 多少？ / 1 Lot = ? | 最小价格变动 / Tick Size | 变动1下=赚/亏多少 / Tick Value |
|------|-------------|-------------|-------------------|
| 黄金 / Gold | 100 盎司 / 100 oz | $0.01 | $1 |
| 白银 / Silver | 5,000 盎司 / 5,000 oz | $0.001 | $5 |
| 原油 / Crude oil | 1,000 桶 / 1,000 bbl | $0.001 | $1 |
| 欧元/美元 / EURUSD | 100,000 欧元 / 100,000 EUR | 0.00001 | $1 |
| 美股(如苹果) / US stocks (e.g. AAPL) | 100 股 / 100 shares | $0.01 | $1 |
| 比特币 / Bitcoin | 1 个 / 1 BTC | $0.01 | $0.01 |
| 股指(道指US30) / Index (Dow US30) | 1 | $0.01 | $0.01 |

## 2.3 爆仓 = 亏到被强制踢出 / 2.3 Liquidation = Forced Out for Losing Too Much

当你的亏损大到一定程度，平台会自动帮你平仓（关闭你的交易），防止你欠平台的钱。

Once your loss crosses a threshold, the broker auto-closes your positions to stop you from owing them money.

平台看的指标叫**预付款比例**：

The metric they watch is the **margin level**:

$$\text{预付款比例 / Margin Level} = \frac{\text{净值 / Equity}}{\text{已用保证金 / Used Margin}} \times 100\%$$

| 预付款比例 / Margin Level | 状态 / Status |
|-----------|------|
| > 100% | 安全，还能继续开仓 / Safe — you can still open new positions |
| < 100% | 危险！不能再开新仓 / Danger — no new positions allowed |
| ≤ 30% | **爆仓!** 系统强制平仓 / **Liquidation!** System force-closes everything |

### 爆仓价公式 / Liquidation-Price Formula

做多（买涨）时，价格跌到多少你会爆仓？

When long (bet on rise), how far can price fall before you are liquidated?

$$\text{爆仓价 / LiqPrice} = \text{开仓价 / EntryPrice} - \frac{\text{净值 / Equity} - \text{保证金 / Margin} \times 30\%}{\text{手数 / Lots} \times \text{合约量 / ContractSize}}$$

做空（买跌）时，价格涨到多少你会爆仓？

When short (bet on fall), how far can price rise before you are liquidated?

$$\text{爆仓价 / LiqPrice} = \text{开仓价 / EntryPrice} + \frac{\text{净值 / Equity} - \text{保证金 / Margin} \times 30\%}{\text{手数 / Lots} \times \text{合约量 / ContractSize}}$$

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import cm
from mpl_toolkits.mplot3d import Axes3D
import ipywidgets as widgets
from ipywidgets import interact, FloatSlider, IntSlider, Dropdown, ToggleButtons
from IPython.display import display, HTML
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 100

# Unified color palette
C_GREEN, C_RED, C_ORANGE, C_BLUE, C_PURPLE = '#2ecc71', '#e74c3c', '#f39c12', '#3498db', '#9b59b6'
print('Setup done! Continue to the next cell.')

## 2.4 互动模拟器 / 2.4 Interactive Simulator

> 选一个产品，调整参数，实时看到：
> - 你需要交多少保证金
> - 价格走势不利时，你的预付款比例怎么下降
> - 到达爆仓线时对应的价格是多少
>
> Pick a product, tweak the sliders, and watch live:
> - how much margin you need to post
> - how your margin level drops when price moves against you
> - the exact price at which you hit the liquidation line

**试试看**：把手数调大，或者把杠杆调小，观察爆仓价怎么变化。

**Try it**: raise the lot size, or lower the leverage, and see how the liquidation price shifts.

In [ ]:
# Product catalog: each entry holds lot contract size, tick size, and a reference mid price.
PRODUCTS = {
    'Gold (100 oz / lot)':        {'contract_size': 100,    'tick_size': 0.01,    'ref_price': 2300},
    'Silver (5000 oz / lot)':     {'contract_size': 5000,   'tick_size': 0.001,   'ref_price': 28},
    'Crude oil (1000 bbl / lot)': {'contract_size': 1000,   'tick_size': 0.001,   'ref_price': 85},
    'EURUSD':                     {'contract_size': 100000, 'tick_size': 0.00001, 'ref_price': 1.08},
    'GBPUSD':                     {'contract_size': 100000, 'tick_size': 0.00001, 'ref_price': 1.25},
    'USDJPY':                     {'contract_size': 100000, 'tick_size': 0.001,   'ref_price': 150},
    'Dow Jones (US30)':           {'contract_size': 1,      'tick_size': 0.01,    'ref_price': 39000},
    'Apple (AAPL)':               {'contract_size': 100,    'tick_size': 0.01,    'ref_price': 180},
    'Bitcoin (BTCUSD)':           {'contract_size': 1,      'tick_size': 0.01,    'ref_price': 63000},
}

def margin_simulator(product='Gold (100 oz / lot)', equity=10000, leverage=100, lots=2.0,
                     liq_level=30, direction='Long (buy)'):
    p = PRODUCTS[product]
    contract_size = p['contract_size']
    price = p['ref_price']

    margin = lots * contract_size * price / leverage
    tick_value = p['tick_size'] * contract_size * lots
    is_long = 'Long' in direction
    if is_long:
        liq_price = price - (equity - margin * liq_level / 100) / (lots * contract_size)
    else:
        liq_price = price + (equity - margin * liq_level / 100) / (lots * contract_size)

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    # --- Left: margin level vs price ---
    ax = axes[0]
    price_range = np.linspace(price * 0.7, price * 1.3, 300)
    if is_long:
        floating_pnl = (price_range - price) * lots * contract_size
    else:
        floating_pnl = (price - price_range) * lots * contract_size
    margin_level = ((equity + floating_pnl) / margin) * 100

    ax.plot(price_range, margin_level, color=C_BLUE, lw=2.5)
    ax.axhline(y=100, color=C_ORANGE, ls='--', lw=1.5, label='100% warning')
    ax.axhline(y=liq_level, color=C_RED, ls='--', lw=1.5, label=f'{liq_level}% liq line')
    ax.fill_between(price_range, 0, margin_level, where=(margin_level <= liq_level), color=C_RED, alpha=0.2)
    ax.fill_between(price_range, 0, margin_level,
                    where=((margin_level > liq_level) & (margin_level <= 100)), color=C_ORANGE, alpha=0.1)
    if 0 < liq_price < price * 2:
        ax.axvline(x=liq_price, color=C_RED, ls=':', alpha=0.7)
        ax.annotate(f'Liquidated!\n{liq_price:.2f}', xy=(liq_price, liq_level),
                    xytext=(liq_price + price * 0.03 * (1 if is_long else -1), liq_level + 25),
                    fontsize=10, fontweight='bold', color=C_RED,
                    arrowprops=dict(arrowstyle='->', color=C_RED, lw=2))
    ax.set_xlabel(f'{product} price', fontsize=11)
    ax.set_ylabel('Margin level (%)', fontsize=11)
    ax.set_title('Margin level vs price', fontsize=13, fontweight='bold')
    ax.legend(fontsize=9)
    ax.set_ylim(0, max(200, margin_level.max() * 1.1))
    ax.grid(True, alpha=0.2)

    # --- Middle: parameter panel ---
    ax = axes[1]
    ax.set_xlim(0, 10); ax.set_ylim(0, 10); ax.axis('off')
    ax.set_title('Your trade parameters', fontsize=13, fontweight='bold')
    margin_pct = margin / equity * 100
    risk_color = C_GREEN if margin_pct < 50 else C_ORANGE if margin_pct < 80 else C_RED
    risk_text = 'SAFE' if margin_pct < 50 else 'Be careful' if margin_pct < 80 else 'VERY DANGEROUS'

    rows = [
        (f'Direction: {direction}', 'black'),
        (f'Margin: ${margin:,.2f}', 'black'),
        (f'% of equity: {margin_pct:.1f}%', risk_color),
        (f'Tick value: ${tick_value:.2f}', 'black'),
        (f'Liq price: {liq_price:.2f}', C_RED),
        (f'Distance: {abs(price - liq_price):.2f} ({abs(price - liq_price) / price * 100:.1f}%)', C_RED),
    ]
    for i, (text, color) in enumerate(rows):
        ax.text(5, 8.5 - i * 1.3, text, ha='center', fontsize=11, color=color,
                bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow', edgecolor='lightgray'))
    ax.text(5, 1.5, risk_text, ha='center', fontsize=18, fontweight='bold', color=risk_color,
            bbox=dict(boxstyle='round,pad=0.5', facecolor=risk_color, alpha=0.15, edgecolor=risk_color))

    # --- Right: P&L curve ---
    ax = axes[2]
    ax.fill_between(price_range, floating_pnl, 0, where=(floating_pnl >= 0), color=C_GREEN, alpha=0.3, label='Profit')
    ax.fill_between(price_range, floating_pnl, 0, where=(floating_pnl < 0), color=C_RED, alpha=0.3, label='Loss')
    ax.plot(price_range, floating_pnl, color='black', lw=2)
    ax.axhline(y=0, color='gray', ls='--')
    ax.axvline(x=price, color=C_BLUE, ls='--', alpha=0.5, label=f'Entry {price}')
    ax.set_xlabel(f'{product} price', fontsize=11)
    ax.set_ylabel('Your P&L (USD)', fontsize=11)
    ax.set_title(f'{direction} P&L curve', fontsize=13, fontweight='bold')
    ax.legend(fontsize=9); ax.grid(True, alpha=0.2)

    plt.tight_layout(); plt.show()

    # Text read-out
    print(f'  You posted ${margin:,.2f} margin to control ${lots * contract_size * price:,.0f} of {product}')
    print(f'  A single tick moves your P&L by ${tick_value:.2f}')
    distance = abs(price - liq_price)
    print(f'  Distance to liquidation: {distance:.2f} ({distance / price * 100:.1f}% room)')
    if margin_pct > 80:
        print(f'  !! Margin is {margin_pct:.0f}% of equity — very risky. Lower lot size or raise leverage.')

interact(margin_simulator,
         product=Dropdown(options=list(PRODUCTS.keys()), value='Gold (100 oz / lot)', description='Product:'),
         equity=IntSlider(value=10000, min=500, max=100000, step=500, description='Capital ($):'),
         leverage=IntSlider(value=100, min=10, max=500, step=10, description='Leverage:'),
         lots=FloatSlider(value=2.0, min=0.1, max=20, step=0.1, description='Lots:'),
         liq_level=IntSlider(value=30, min=10, max=80, step=5, description='Liq line (%):'),
         direction=ToggleButtons(options=['Long (buy)', 'Short (sell)'], description='Direction:'));

## 2.5 小结 / 2.5 Chapter Recap

| 你需要记住的 / Must Remember | 公式 / Formula |
|------------|------|
| 保证金 / Margin | 手数 × 合约量 × 价格 ÷ 杠杆 / Lots × ContractSize × Price ÷ Leverage |
| 点值 / Tick value | 最小波动 × 合约量 × 手数 / TickSize × ContractSize × Lots |
| 预付款比例 / Margin level | 净值 ÷ 保证金 × 100% / Equity ÷ Margin × 100% |
| 爆仓价(多) / Liq price (long) | 开仓价 - (净值-保证金×30%) ÷ (手数×合约量) / Entry − (Equity − Margin × 30%) ÷ (Lots × ContractSize) |
| 爆仓价(空) / Liq price (short) | 开仓价 + (净值-保证金×30%) ÷ (手数×合约量) / Entry + (Equity − Margin × 30%) ÷ (Lots × ContractSize) |

> **关键理解**：杠杆越高 → 保证金越少 → 可以开更大仓位 → 但爆仓风险也更大
>
> **Key insight**: higher leverage → smaller margin → bigger positions allowed → but much bigger liquidation risk.
>
> **下一章：交易成本** —— 每笔交易你要付哪些费用？
>
> **Next chapter: Trading Costs** — what fees you actually pay on every trade.